# 01 — Data Check
Verify that downloaded CPC/IMERG data looks correct before running the full pipeline.

**Kernel:** Python (atmo)

In [ ]:
import sys
sys.path.insert(0, '..')  # add project root so 'src' is importable

import yaml
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Load config
with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config loaded.')
print(f"  Years: {cfg['years']['start']}–{cfg['years']['end']}")
print(f"  Wet threshold: {cfg['analysis']['wet_threshold']} mm/day")
print(f"  Rolling window: {cfg['analysis']['rolling_days']} days")

## 1. Check CPC data

In [ ]:
from src.data.preprocess import load_cpc

# Load a single test year to check structure
test_cfg = dict(cfg)
test_cfg['years'] = {'start': 2010, 'end': 2010}

da_cpc = load_cpc(test_cfg)
print(da_cpc)
print(f"\nShape : {da_cpc.shape}")
print(f"Lat   : {float(da_cpc.lat.min()):.2f} – {float(da_cpc.lat.max()):.2f}")
print(f"Lon   : {float(da_cpc.lon.min()):.2f} – {float(da_cpc.lon.max()):.2f}")
print(f"Range : {float(da_cpc.min()):.2f} – {float(da_cpc.max()):.2f} mm/day")
print(f"NaN % : {100 * float(np.isnan(da_cpc).mean()):.1f}%")

In [ ]:
# Plot a single day
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    print('cartopy not installed — using basic imshow')

day = da_cpc.isel(time=100)  # pick one day
title = f"CPC Daily Precip — {str(da_cpc.time.values[100])[:10]}"

if HAS_CARTOPY:
    fig, ax = plt.subplots(1, 1, figsize=(10, 5),
                           subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent([-125, -66, 24, 50])
    ax.add_feature(cfeature.STATES, linewidth=0.5)
    ax.add_feature(cfeature.COASTLINE)
    im = ax.pcolormesh(day.lon, day.lat, day,
                       transform=ccrs.PlateCarree(),
                       cmap='Blues', vmin=0, vmax=50)
    fig.colorbar(im, ax=ax, label='mm/day')
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(day, origin='lower', cmap='Blues', vmin=0, vmax=50)
    fig.colorbar(im, ax=ax, label='mm/day')

ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of non-zero values on this day
vals = day.values.ravel()
vals = vals[np.isfinite(vals) & (vals > 0)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(vals, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_xlabel('Precip (mm/day)')
axes[0].set_ylabel('Count')
axes[0].set_title('CPC: daily values (wet grid points)')

axes[1].hist(np.log1p(vals), bins=60, color='coral', edgecolor='white', linewidth=0.3)
axes[1].set_xlabel('log(1 + precip)')
axes[1].set_title('CPC: log-scale (hint of log-normality?)')

plt.tight_layout()
plt.show()
print(f"Wet grid points: {len(vals)}   Mean: {vals.mean():.2f}   Std: {vals.std():.2f}")

## 2. Check preprocessed data (7-day rolling mean)

In [ ]:
from src.data.preprocess import compute_rolling, mask_wet_days

da_rolled = compute_rolling(da_cpc, test_cfg)
da_wet    = mask_wet_days(da_rolled, test_cfg)

print(f"After 7-day rolling + masking:")
print(f"  Shape: {da_wet.shape}")
print(f"  Wet values: {int(np.isfinite(da_wet).sum()):,}")
print(f"  Wet fraction: {float(np.isfinite(da_wet).mean()):.3f}")
print(f"  Range: {float(da_wet.min()):.2f} – {float(da_wet.max()):.2f} mm/day")

In [ ]:
# Time series at a sample point (e.g., Kansas City area)
lat_pt, lon_pt = 39.0, -94.5

ts_raw    = da_cpc.sel(lat=lat_pt, lon=lon_pt, method='nearest')
ts_rolled = da_rolled.sel(lat=lat_pt, lon=lon_pt, method='nearest')

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(ts_raw)), ts_raw.values, width=1, color='steelblue', alpha=0.5, label='Daily')
ax.plot(ts_rolled.values, color='red', lw=1.5, label='7-day rolling mean')
ax.axhline(cfg['analysis']['wet_threshold'], color='orange', ls='--', lw=1, label='Wet threshold')
ax.set_xlabel('Day of year 2010')
ax.set_ylabel('Precip (mm/day)')
ax.set_title(f'CPC at lat={lat_pt}, lon={lon_pt} (Kansas City area)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Quick check for IMERG (if available)

In [ ]:
import os
from pathlib import Path

imerg_dir = Path('../data/raw/imerg')
imerg_files = list(imerg_dir.glob('*.nc4'))

if imerg_files:
    print(f'IMERG files found: {len(imerg_files)}')
    print(f'First: {imerg_files[0].name}')
    # Load one file to check
    ds_test = xr.open_dataset(str(imerg_files[0]))
    print(ds_test)
else:
    print('No IMERG files yet. Run download_imerg first.')
    print('(You need a NASA Earthdata account — see README.md)')